In [ ]:
#PERTEMUAN 6
#Nomor 1: Analisis Speedup Skala


import time
import multiprocessing as mp
import matplotlib.pyplot as plt
import numpy as np

# Fungsi pembantu untuk mengecek bilangan prima
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# Fungsi eksperimen 7.1 (hitung_prima)
def hitung_prima_range(start_end):
    start, end = start_end
    count = 0
    for num in range(start, end):
        if is_prime(num):
            count += 1
    return count

def run_experiment_1():
    limit = 2000000
    worker_counts = [1, 2, 4, 8]
    times = []

    # Membagi rentang kerja menjadi 8 chunk agar seimbang
    chunk_size = limit // 8
    ranges = [(i * chunk_size, (i + 1) * chunk_size) for i in range(8)]

    # Menggunakan mp.get_context('fork').Pool agar konsisten
    ctx = mp.get_context('fork')

    print("--- Mengukur Waktu Eksekusi ---")
    for w in worker_counts:
        start_time = time.time()
        with ctx.Pool(processes=w) as pool:
            results = pool.map(hitung_prima_range, ranges)
        elapsed = time.time() - start_time
        times.append(elapsed)
        print(f"Worker = {w}: Waktu = {elapsed:.4f} detik")

    # Kalkulasi Speedup
    t1 = times[0]
    actual_speedup = [t1 / t for t in times]

    # Hukum Amdahl Teoritis (Asumsi p = 0.95 atau 95% kode paralel)
    p = 0.95
    amdahl_speedup = [1 / ((1 - p) + (p / w)) for w in worker_counts]

    # Plotting
    plt.figure(figsize=(8, 5))
    plt.plot(worker_counts, actual_speedup, 'o-', label='Actual Speedup', color='blue')
    plt.plot(worker_counts, amdahl_speedup, 's--', label='Amdahl Theory (p=0.95)', color='red')
    plt.plot(worker_counts, worker_counts, 'k:', label='Ideal Linear Speedup')
    plt.xlabel('Jumlah Worker (Processes)')
    plt.ylabel('Speedup')
    plt.title('Speedup Skala vs Hukum Amdahl')
    plt.legend()
    plt.grid(True)
    plt.show()

if __name__ == '__main__':
    run_experiment_1()


In [ ]:
Penjelasan dan Analisis Hasil
Kurva aktual mungkin tidak persis mengikuti atau bahkan menyimpang dari kurva teoritis karena beberapa faktor utama:
Jumlah Core Fisik Mesin: Jika jumlah worker melebihi jumlah core fisik CPU mesin, proses akan saling berebut waktu eksekusi (context switching overhead) sehingga tidak terjadi peningkatan kecepatan yang signifikan.

Overhead Pembuatan Proses: Membuat proses baru (fork) dan menyalurkan data antar-proses memerlukan alokasi memori serta manajemen OS tersendiri yang menambah time delay.

Beban Kerja Lain (Background Load): Proses sistem operasi, aplikasi latar belakang, atau garbage collection dapat menggunakan sumber daya CPU saat eksperimen berjalan.

Ketidakseimbangan Beban (Load Imbalance): Pengecekan bilangan prima pada angka yang lebih besar membutuhkan iterasi modulasi yang lebih banyak dibandingkan angka kecil.

In [ ]:
#2.Nomor 2: Shared Memory vs Message Passing

import time
import random
import multiprocessing as mp

N_PER_PROCESS = 1_000_000
NUM_PROCESSES = 4

# (a) Versi Queue (Message Passing)
def worker_queue(queue):
    # Setiap proses menjumlahkan 1 juta angka acak secara independen
    subtotal = sum(random.random() for _ in range(N_PER_PROCESS))
    queue.put(subtotal)

# (b) Versi Value (Shared Memory dengan Lock)
def worker_value(shared_val, lock):
    # Setiap proses langsung menambah ke shared value menggunakan Lock
    for _ in range(N_PER_PROCESS):
        val = random.random()
        with lock:
            shared_val.value += val

def run_experiment_2():
    # --- 1. Uji Versi Queue ---
    queue = mp.Queue()
    processes = []
    start_time = time.time()

    for _ in range(NUM_PROCESSES):
        p = mp.Process(target=worker_queue, args=(queue,))
        processes.append(p)
        p.start()

    subtotals = [queue.get() for _ in range(NUM_PROCESSES)]
    total_queue = sum(subtotals)

    for p in processes:
        p.join()

    time_queue = time.time() - start_time

    # --- 2. Uji Versi Value (Shared Memory + Lock) ---
    shared_val = mp.Value('d', 0.0)
    lock = mp.Lock()
    processes = []
    start_time = time.time()

    for _ in range(NUM_PROCESSES):
        p = mp.Process(target=worker_value, args=(shared_val, lock))
        processes.append(p)
        p.start()

    for p in processes:
        p.join()

    time_value = time.time() - start_time

    print(f"Versi Queue (Message Passing) Time : {time_queue:.4f} detik (Total: {total_queue:.2f})")
    print(f"Versi Value (Shared Memory) Time   : {time_value:.4f} detik (Total: {shared_val.value:.2f})")

if __name__ == '__main__':
    run_experiment_2()

In [ ]:
Versi Queue (Message Passing) jauh lebih cepat karena setiap proses menghitung akumulasi lokalnya di memori privat terlebih dahulu, lalu hanya mengirimkan 1 hasil akhir (subtotal) melalui Queue.Versi Value (Shared Memory + Lock) jauh lebih lambat karena lock dipanggil sebanyak $4 \times 1.000.000 = 4.000.000$ kali. Tingginya contention dan overhead sinkronisasi lock pada setiap iterasi kecil menghancurkan efek paralelisasi.

In [1]:

#Nomor 3: Producer-Consumer Antar-Prosesimport time
import multiprocessing as mp

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

# Producer: Mengirimkan tugas angka (CPU-bound workload)
def producer(queue, task_count):
    for i in range(100000, 100000 + task_count):
        queue.put(i)
    # Kirim sinyal Sentinel (None) untuk menghentikan consumer
    queue.put(None)

# Consumer: Memproses perhitungan CPU-bound
def consumer(queue, result_queue):
    while True:
        item = queue.get()
        if item is None:
            break
        # Komputasi CPU-bound
        result = is_prime(item)
        result_queue.put((item, result))

def run_experiment_3():
    task_queue = mp.Queue()
    result_queue = mp.Queue()

    p_prod = mp.Process(target=producer, args=(task_queue, 500))
    p_cons = mp.Process(target=consumer, args=(task_queue, result_queue))

    start_time = time.time()
    p_prod.start()
    p_cons.start()

    p_prod.join()
    p_cons.join()
    elapsed = time.time() - start_time

    print(f"Selesai memproses producer-consumer dalam {elapsed:.4f} detik.")
    print(f"Jumlah hasil yang didapat: {result_queue.qsize()}")

if __name__ == '__main__':
    run_experiment_3()

Selesai memproses producer-consumer dalam 0.1495 detik.
Jumlah hasil yang didapat: 0


In [ ]:
4.Refleski
saya menggunakan Manager() saat membutuhkan struktur data kompleks (seperti list atau dict) yang fleksibel dan perlu diakses bersama oleh banyak proses tanpa mengelola komputasi memori manual. Contohnya adalah menyimpan hasil pencarian log dari berbagai server ke dalam satu dictionary bersama.

Sebaliknya, saya menggunakan Value / Array untuk variabel atau array primitif bertipe data tetap (seperti int atau float) dengan ukuran terdefinisikan, contohnya memperbarui counter transaksi berkinerja tinggi.

Trade-off: Manager() unggul dari segi kemudahan karena mendukung manipulasi tipe data tingkat tinggi melalui proxy objects, namun memiliki performa lebih lambat akibat overhead proses server terpisah dan IPC serialization. Value/Array memberikan performa jauh lebih cepat karena berbasis shared memory langsung di RAM, meskipun kurang fleksibel dari segi tipe data.

In [3]:
#PERETMUAN 5
#soal 1
import threading
import time

# Class Rekening Bank Sederhana
class BankAccount:
    def __init__(self, initial_balance=1_000_000):
        self.balance = initial_balance
        self.lock = threading.Lock()

    # Operasi tanpa Lock (Rentan Race Condition)
    def transaksi_tanpa_lock(self):
        for _ in range(10_000):
            # Tarik 1 lalu setor 1
            temp = self.balance
            temp -= 1
            temp += 1
            self.balance = temp

    # Operasi dengan Lock (Aman/Thread-Safe)
    def transaksi_dengan_lock(self):
        for _ in range(10_000):
            with self.lock:
                temp = self.balance
                temp -= 1
                temp += 1
                self.balance = temp

# --- 1. Simulasi Tanpa Lock ---
acc_no_lock = BankAccount()
threads_1 = [threading.Thread(target=acc_no_lock.transaksi_tanpa_lock) for _ in range(10)]

for t in threads_1:
    t.start()
for t in threads_1:
    t.join()

print(f"Saldo Akhir Tanpa Lock  : {acc_no_lock.balance:,} (Ekspektasi: 1,000,000)")

# --- 2. Simulasi Dengan Lock ---
acc_with_lock = BankAccount()
threads_2 = [threading.Thread(target=acc_with_lock.transaksi_dengan_lock) for _ in range(10)]

for t in threads_2:
    t.start()
for t in threads_2:
    t.join()

print(f"Saldo Akhir Dengan Lock : {acc_with_lock.balance:,} (Ekspektasi: 1,000,000)")

Saldo Akhir Tanpa Lock  : 1,000,000 (Ekspektasi: 1,000,000)
Saldo Akhir Dengan Lock : 1,000,000 (Ekspektasi: 1,000,000)


In [4]:
#SOAL 2
import queue
import threading
import time
import random

# Producer mengirim angka ke queue
def producer(q, num_items):
    for i in range(1, num_items + 1):
        item = f"Data-{i}"
        q.put(item)
        print(f"[Producer] Memasukkan {item}")
        time.sleep(0.01)
    
    # Mengirimkan sinyal None sebanyak jumlah consumer (3 consumer)
    print("[Producer] Selesai memproduksi. Mengirim sinyal None ke seluruh consumer...")
    for _ in range(3):
        q.put(None)

# Consumer mengambil data dari queue
def consumer(q, consumer_id):
    while True:
        item = q.get()
        if item is None:
            print(f"  [Consumer-{consumer_id}] Menerima sinyal 'None'. Meringkas dan berhenti.")
            q.task_done()
            break
        
        # Simulasi pemrosesan kerja
        print(f"  [Consumer-{consumer_id}] Memproses {item}")
        time.sleep(0.02)
        q.task_done()

# Inisialisasi Queue dan Thread
q = queue.Queue()
NUM_ITEMS = 15

t_producer = threading.Thread(target=producer, args=(q, NUM_ITEMS))
consumers = [threading.Thread(target=consumer, args=(q, i+1)) for i in range(3)]

# Jalankan seluruh Thread
t_producer.start()
for c in consumers:
    c.start()

# Tunggu seluruh proses selesai
t_producer.join()
for c in consumers:
    c.join()

print("Seluruh producer dan 3 consumer telah selesai secara bersih.")

[Producer] Memasukkan Data-1
  [Consumer-1] Memproses Data-1
[Producer] Memasukkan Data-2
  [Consumer-2] Memproses Data-2
[Producer] Memasukkan Data-3
  [Consumer-3] Memproses Data-3
[Producer] Memasukkan Data-4
  [Consumer-1] Memproses Data-4
[Producer] Memasukkan Data-5
  [Consumer-2] Memproses Data-5
[Producer] Memasukkan Data-6
  [Consumer-3] Memproses Data-6
[Producer] Memasukkan Data-7
  [Consumer-1] Memproses Data-7
[Producer] Memasukkan Data-8
  [Consumer-2] Memproses Data-8
[Producer] Memasukkan Data-9
  [Consumer-3] Memproses Data-9
[Producer] Memasukkan Data-10
  [Consumer-1] Memproses Data-10
[Producer] Memasukkan Data-11
  [Consumer-2] Memproses Data-11
[Producer] Memasukkan Data-12
  [Consumer-3] Memproses Data-12
[Producer] Memasukkan Data-13
  [Consumer-1] Memproses Data-13
[Producer] Memasukkan Data-14
  [Consumer-2] Memproses Data-14
[Producer] Memasukkan Data-15
  [Consumer-3] Memproses Data-15
[Producer] Selesai memproduksi. Mengirim sinyal None ke seluruh consumer.

In [ ]:
#SOAL 3

import concurrent.futures
import time
import matplotlib.pyplot as plt
import urllib.request

# Fungsi simulasi I/O bound (Download file dummy / fetch URL)
def download_file(file_id):
    # Menggunakan httpbin untuk simulasi delay I/O request
    url = f"https://httpbin.org/delay/0.1?id={file_id}"
    try:
        with urllib.request.urlopen(url, timeout=5) as response:
            return response.status
    except Exception as e:
        return str(e)

def run_experiment_3():
    total_files = 50
    worker_options = [2, 4, 8, 16]
    times = []

    file_ids = list(range(1, total_files + 1))

    print("--- Mengukur Waktu Eksekusi ThreadPoolExecutor ---")
    for workers in worker_options:
        start_time = time.time()
        with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
            results = list(executor.map(download_file, file_ids))
        elapsed = time.time() - start_time
        times.append(elapsed)
        print(f"max_workers = {workers:2d} | Waktu Eksekusi: {elapsed:.2f} detik")

    # Plot Waktu Eksekusi vs max_workers
    plt.figure(figsize=(8, 5))
    plt.plot(worker_options, times, 'o-', color='purple', linewidth=2, markersize=8)
    plt.title('Eksperimen ThreadPoolExecutor (50 File I/O)')
    plt.xlabel('Jumlah max_workers')
    plt.ylabel('Waktu Eksekusi (detik)')
    plt.grid(True)
    plt.xticks(worker_options)
    plt.show()

if __name__ == '__main__':
    run_experiment_3()

--- Mengukur Waktu Eksekusi ThreadPoolExecutor ---
max_workers =  2 | Waktu Eksekusi: 31.35 detik
max_workers =  4 | Waktu Eksekusi: 15.46 detik


In [ ]:
SOAL 4

Contoh Kasus Membantu (I/O-Bound): Web Scraper multi-halaman atau bot notifier. Ketika program menunggu respons HTTP dari server atau transaksi basis data, GIL dilepas sehingga thread lain dapat langsung memproses request berikutnya secara bersamaan tanpa membuang waktu tunggu CPU.

Contoh Kasus Tidak Membantu (CPU-Bound): Rendering citra/video digital atau pemrosesan machine learning secara manual. Tugas ini membutuhkan siklus CPU murni tanpa waktu henti I/O. Di Python, Global Interpreter Lock (GIL) mencegah beberapa thread mengeksekusi instruksi bytecode Python secara sejajar di banyak core CPU, sehingga threading justru memperlambat performa akibat overhead context switching. Kasus ini membutuhkan modul multiprocessing.